In [ ]:
import os
import csv
from collections import defaultdict

# Token class
class Token:
    def __init__(self, id_, form, lemma, upos, xpos, feats, head, deprel, deps, misc):
        self.id = id_
        self.form = form
        self.lemma = lemma
        self.upos = upos
        self.xpos = xpos
        self.feats = feats if feats is not None else {}
        self.head = head
        self.deprel = deprel
        self.deps = deps
        self.misc = misc

class Sentence:
    def __init__(self, sentence_id, tokens, file_name, metadata=None):
        self.sentence_id = sentence_id
        self.tokens = tokens
        self.file_name = file_name  # Store file name
        self.metadata = metadata if metadata else {}

    def get_tokens(self):
        return self.tokens

    def get_metadata(self):
        return self.metadata

    def get_file_name(self):
        return self.file_name  # Method to get the file name

def process_csv_file(file_path):
    sentences = []
    current_tokens = []
    current_metadata = {}
    current_sentence_id = None
    file_name = os.path.basename(file_path)  # Get the file name from the path

    with open(file_path, 'r', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile, delimiter='\t')
        for row in reader:
            if not row:
                continue

            # Detect sentence ID
            if row[0].startswith("#SENTENCE ID"):
                if current_sentence_id is not None:
                    sentences.append(Sentence(current_sentence_id, current_tokens, file_name, current_metadata))
                
                current_sentence_id = row[0].split("=")[1].strip()
                current_tokens = []
                current_metadata = {}
            elif row[0].startswith("#"):
                if "=" in row[0]:
                    key, value = row[0].lstrip("#").split("=", 1)
                    current_metadata[key.strip()] = value.strip()
            else:
                token = Token(
                    id_=row[0],
                    form=row[1],
                    lemma=row[9],
                    upos=row[12],
                    xpos=row[13],
                    feats=row[13] if len(row) > 13 else "",  # Updated index for feats
                    head=row[2],
                    deprel=row[3],
                    deps=row[4],
                    misc=row[11] if len(row) > 17 else ""  # Ensure misc is also checked for missing data
                )
                current_tokens.append(token)

        if current_sentence_id is not None:
            sentences.append(Sentence(current_sentence_id, current_tokens, file_name, current_metadata))

    return sentences

# Process all CSV files in a folder
def process_folder(folder_path):
    all_sentences = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".csv"):
                file_path = os.path.join(root, file)
                sentences = process_csv_file(file_path)
                all_sentences.extend(sentences)
    return all_sentences

def find_nouns_with_degree_cmp_grouped(sentences):
    grouped_nouns = []
    
    for sentence in sentences:
        for token in sentence.get_tokens():  # Access tokens correctly
            if token.upos == 'NOUN' and token.feats != '_':
                feats = token.feats.strip()  # Remove leading/trailing spaces
                
                # Split feats by '|' and filter out invalid "key=value" pairs
                valid_feats = [f for f in feats.split('|') if '=' in f and f.count('=') == 1]
                feats_dict = dict(f.split('=') for f in valid_feats)
                
                # Only process if there's a valid degree comparison feature
                if 'Degree' in feats_dict:
                    degree = feats_dict['Degree']
                    if degree:
                        # Prepare data for output, now excluding the Newpart field
                        grouped_nouns.append({
                            'Sentence ID': sentence.sentence_id,
                            'File Name': sentence.get_file_name(),  # Include file name
                            'Token ID': token.id,
                            'Form': token.form,
                            'Lemma': token.lemma,
                            'UPOS': token.upos,
                            'XPOS': token.xpos,
                            'Features': token.feats,
                            'Head': token.head,
                            'Deprel': token.deprel,
                            'Deps': token.deps,
                            'Misc': token.misc
                        })
    
    return grouped_nouns

def save_to_csv(data, output_file):
    fieldnames = ['Sentence ID', 'File Name', 'Token ID', 'Form', 'Lemma', 'UPOS', 'XPOS', 'Features', 'Head', 'Deprel', 'Deps', 'Misc']
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)  # Ensure the data is a list of dictionaries
        print(f"Results saved to {output_file}")
    except Exception as e:
        print(f"Error writing to CSV: {e}")

# Example usage
def example_usage():
    folder_path = 'C:\\Users\\rahaa\\Dropbox\\MPCD\\codes\\98-syntactically_annotated_to_import'
    output_file = 'nouns_with_degree_cmp.csv'

    sentences = process_folder(folder_path)
    nouns_with_degree_cmp = find_nouns_with_degree_cmp_grouped(sentences)
    save_to_csv(nouns_with_degree_cmp, output_file)

if __name__ == "__main__":
    example_usage()


1473
Results saved to nominalized_adjectives.csv
